# Model Prototyping Notebook

This notebook uses the repository scripts and data layout to prototype machine learning models for the dedicated workflow.

In [8]:
from pathlib import Path
import sys

notebook_dir = Path.cwd().resolve()
repo_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir

try:
    import pandas as pd
    import numpy as np
except ModuleNotFoundError:
    %pip install -r "{repo_root / 'requirements.txt'}"
    import pandas as pd
    import numpy as np

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

repo_root

WindowsPath("C:/Users/Nasim/OneDrive/Desktop/COMPAS Research Datasets/IEEE COMPAS '26")

In [9]:
# Load project documentation and source files for context
for filename in ['README.md', 'ARCHITECTURE.md', 'requirements.txt']:
    path = repo_root / filename
    if path.exists():
        print(f'\n===== {filename} =====')
        print(path.read_text(encoding='utf-8'))


===== README.md =====
# Beyond Accuracy: Diagnosing Systematic Biases in Classical ML Weather Models Using SHAP Explainability

This repository contains the replication code and experimental pipeline for auditing classical machine learning weather models predicting precipitation and near-surface air temperature (T2M) for Dhaka, Bangladesh, using 45 years (1981–2025) of NASA POWER reanalysis data.

## Project Architecture
* `data/`: Contains raw climate matrix data and formatted time-series outputs.
* `src/`: Modular Python pipeline implementing data engineering, model suites, and TreeSHAP diagnostics.
* `figures/`: Publication-grade, 300 DPI visual outputs (Beeswarm and seasonal dependence plots).

## Installation & Setup
1. Clone the repository and navigate to the project root.
2. Install the necessary dependencies:
   ```bash
   pip install -r requirements.txt

===== ARCHITECTURE.md =====
IEEE COMPAS '26/
│
├── data/
│   ├── raw/                 # Store the original 'data.csv' here 

In [10]:
# Import project modules
import data_preprocessing as dp
import model_training as mt
import utils as u

print('Imported modules successfully.')

Imported modules successfully.


In [11]:
# Check for processed data
processed_path = repo_root / 'data' / 'processed' / 'df_processed.csv'
processed_path.exists()

True

In [12]:
# Load processed data if available
if processed_path.exists():
    df = pd.read_csv(processed_path, parse_dates=['DATE'], index_col='DATE')
    print(df.head())
    print('\nColumns:', list(df.columns))
else:
    print('Processed data not found. Run the preprocessing script first.')

            ALLSKY_SFC_SW_DWN  CLOUD_AMT  PRECTOTCORR      PS   QV2M    T2M  \
DATE                                                                          
1984-01-01             4.0250      26.21         0.11  101.28   5.43  18.71   
1984-02-01             4.6531      34.29         0.01  101.13   4.71  21.31   
1984-03-01             5.7031      30.38         0.01  100.71   6.38  28.18   
1984-04-01             5.7763      58.32         0.75  100.34  12.87  32.43   
1984-05-01             5.1067      72.58         8.27  100.10  18.15  30.17   

               TS  WS10M_MIN   WSC       SEASON  T2M_LAG_1  PRECTOTCORR_LAG_1  
DATE                                                                           
1984-01-01  18.60       0.21  1.76          Dry      19.33               0.20  
1984-02-01  21.90       0.23  2.12          Dry      18.71               0.11  
1984-03-01  28.47       0.30  2.06  Pre-Monsoon      21.31               0.01  
1984-04-01  34.07       0.19  2.56  Pre-Monsoo

In [13]:
# Inspect the target columns and feature layout
if 'df' in locals():
    target_candidates = [col for col in df.columns if col in {'T2M', 'PRECTOTCORR', 'TARGET', 'label'}]
    print('Target candidates:', target_candidates)
    print('Feature columns:', [col for col in df.columns if col not in target_candidates and col != 'SEASON'])

Target candidates: ['PRECTOTCORR', 'T2M']
Feature columns: ['ALLSKY_SFC_SW_DWN', 'CLOUD_AMT', 'PS', 'QV2M', 'TS', 'WS10M_MIN', 'WSC', 'T2M_LAG_1', 'PRECTOTCORR_LAG_1']


In [14]:
# Create a simple train/test split when a target exists
if 'df' in locals() and target_candidates:
    target_name = target_candidates[0]
    train = df[df.index.year < 2015]
    test = df[df.index.year >= 2015]
    print('Train rows:', len(train))
    print('Test rows:', len(test))
    print('Target selected:', target_name)

Train rows: 372
Test rows: 132
Target selected: PRECTOTCORR


In [15]:
# Quick prototype training example using a simple model
if 'df' in locals() and target_candidates:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.preprocessing import StandardScaler

    target_name = target_candidates[0]
    feature_columns = [col for col in df.columns if col not in {target_name, 'SEASON'}]
    X_train = train[feature_columns]
    y_train = train[target_name]
    X_test = test[feature_columns]
    y_test = test[target_name]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    print(f'MAE: {mae:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2: {r2:.4f}')

MAE: 2.4126
RMSE: 3.6476
R2: 0.6667


In [16]:
# Wrap up with next steps
print('Next steps:')
print('- Run the preprocessing pipeline to generate df_processed.csv')
print('- Expand the model suite using the scripts in src/')
print('- Save figures and metrics for the manuscript')

Next steps:
- Run the preprocessing pipeline to generate df_processed.csv
- Expand the model suite using the scripts in src/
- Save figures and metrics for the manuscript
